In [1]:
# -*- coding: utf-8 -*-
"""CALCULODEVOLATILIDADE-EGARCH.ipynb"""

import pandas as pd, numpy as np, math, os
import scipy.optimize as opt
from scipy.stats import norm
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment, PatternFill

path="/content/PREÇO MED DIARIO PLD VERTICAL PERIODO 17-04-18 A 03-06-25.xlsx"
df=pd.read_excel(path)

# detect columns
cols=list(df.columns)
date_col=None; price_col=None

for c in cols:
    cl=str(c).strip().lower()
    if date_col is None and "data" in cl:
        date_col=c
    if price_col is None and ("preço" in cl or "preco" in cl or "pld" in cl):
        if "preço" in cl or "preco" in cl or "preço medio" in cl or "preco medio" in cl:
            price_col=c

if date_col is None:
    for c in cols:
        if np.issubdtype(df[c].dtype, np.datetime64):
            date_col=c; break

if price_col is None:
    for c in cols:
        if np.issubdtype(df[c].dtype, np.number):
            price_col=c; break

df=df.rename(columns={date_col:"data", price_col:"preço_PLD"})
df["data"]=pd.to_datetime(df["data"])
df=df.sort_values("data").reset_index(drop=True)

df["retorno_log"]=np.log(df["preço_PLD"]/df["preço_PLD"].shift(1))
df["volatilidade_realizada_diária"]=df["retorno_log"].abs()

# returns em %
r = df["retorno_log"].dropna().values*100.0
T=len(r)

Eabs = math.sqrt(2/math.pi)

def egarch_negloglik(theta_raw, r):

    mu = theta_raw[0]
    omega = theta_raw[1]

    alpha = math.exp(theta_raw[2])
    gamma = theta_raw[3]
    beta  = math.tanh(theta_raw[4])

    eps = r - mu

    logh = np.empty_like(eps)
    h = np.empty_like(eps)
    z = np.empty_like(eps)

    h0 = max(np.var(eps), 1e-6)

    logh[0] = math.log(h0)
    h[0] = h0
    z[0] = eps[0]/math.sqrt(h0)

    for t in range(1, len(eps)):

        logh[t] = omega + beta*logh[t-1] + alpha*(abs(z[t-1]) - Eabs) + gamma*z[t-1]

        if logh[t] < -50:
            logh[t] = -50
        elif logh[t] > 50:
            logh[t] = 50

        h[t] = math.exp(logh[t])
        z[t] = eps[t]/math.sqrt(h[t])

    ll = -0.5*(np.log(2*math.pi) + np.log(h) + (eps**2)/h)

    return -np.sum(ll)


# estimação
mu0=np.mean(r)
var0=np.var(r)

starts = [
    np.array([mu0, math.log(var0), math.log(0.1), 0.0, math.atanh(0.9)]),
    np.array([mu0, math.log(var0*0.5), math.log(0.2), -0.05, math.atanh(0.95)]),
    np.array([mu0, math.log(var0*1.5), math.log(0.05), 0.05, math.atanh(0.8)]),
]

best=None

for x0 in starts:

    r1=opt.minimize(egarch_negloglik, x0, args=(r,), method="Powell",
                    options={"maxiter":6000})

    x=r1.x

    r2=opt.minimize(egarch_negloglik, x, args=(r,), method="L-BFGS-B",
                    options={"maxiter":8000})

    cand = r2 if r2.fun <= r1.fun else r1

    if best is None or cand.fun < best.fun:
        best=cand


theta=best.x

mu = theta[0]
omega = theta[1]
alpha = math.exp(theta[2])
gamma = theta[3]
beta  = math.tanh(theta[4])


# filtrar volatilidade diária
r_full = df["retorno_log"].values*100.0
eps_full = r_full - mu

N=len(df)

logh=np.full(N, np.nan)
h=np.full(N, np.nan)
z=np.full(N, np.nan)

first = np.where(~np.isnan(eps_full))[0][0]

h0 = max(np.nanvar(eps_full), 1e-6)

logh[first]=math.log(h0)
h[first]=h0
z[first]=eps_full[first]/math.sqrt(h0)

for t in range(first+1, N):

    if np.isnan(eps_full[t-1]) or np.isnan(z[t-1]) or np.isnan(logh[t-1]):
        logh[t]=logh[t-1]
        h[t]=h[t-1]
        z[t]=np.nan if np.isnan(eps_full[t]) else eps_full[t]/math.sqrt(h[t])
        continue

    logh[t]=omega + beta*logh[t-1] + alpha*(abs(z[t-1])-Eabs) + gamma*z[t-1]

    logh[t]=min(max(logh[t], -50), 50)

    h[t]=math.exp(logh[t])

    z[t]=np.nan if np.isnan(eps_full[t]) else eps_full[t]/math.sqrt(h[t])


# volatilidade diária
df["volatilidade_EGARCH_diária"]=np.sqrt(h)/100.0


# =============================
# NOVA ESCALA TEMPORAL
# =============================

for n, lab in [(5,"1W"),(21,"1M"),(252,"1Y")]:

    df[f"volatilidade_EGARCH_{lab}"] = (
        df["volatilidade_EGARCH_diária"] * math.sqrt(n)
    )

    df[f"volatilidade_EGARCH_{lab}_anualizada"] = (
        df["volatilidade_EGARCH_diária"] * math.sqrt(252)
    )


out=df[[
"data",
"preço_PLD",
"retorno_log",
"volatilidade_realizada_diária",
"volatilidade_EGARCH_diária",
"volatilidade_EGARCH_1W",
"volatilidade_EGARCH_1M",
"volatilidade_EGARCH_1Y",
"volatilidade_EGARCH_1W_anualizada",
"volatilidade_EGARCH_1M_anualizada",
"volatilidade_EGARCH_1Y_anualizada"
]].copy()


out_path="/content/PLD_volatilidades_EGARCH.xlsx"

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:

    out.to_excel(writer, index=False, sheet_name="EGARCH")

    params=pd.DataFrame({
        "Item":[
            "Modelo (EGARCH(1,1))",
            "Equação (log-variância)",
            "Erro",
            "E|z| (Normal)",
            "mu",
            "omega",
            "alpha",
            "gamma",
            "beta"
        ],

        "Valor":[
            "log(h_t)=omega + beta*log(h_{t-1}) + alpha*(|z_{t-1}|-E|z|) + gamma*z_{t-1}",
            "h_t = sigma_t² ; z_t = eps_t/sigma_t ; eps_t = r_t - mu",
            "Normal",
            Eabs,
            mu, omega, alpha, gamma, beta
        ]
    })

    params.to_excel(writer, index=False, sheet_name="Parâmetros")


wb=load_workbook(out_path)

header_fill=PatternFill("solid", fgColor="1F4E79")
header_font=Font(color="FFFFFF", bold=True)

ws=wb["EGARCH"]

ws.freeze_panes="A2"

for j,cell in enumerate(ws[1], start=1):

    cell.fill=header_fill
    cell.font=header_font
    cell.alignment=Alignment(horizontal="center", vertical="center", wrap_text=True)

    ws.column_dimensions[get_column_letter(j)].width=24 if j>1 else 14


for cell in ws["A"][1:]:
    cell.number_format="dd/mm/yyyy"


ws2=wb["Parâmetros"]

ws2.freeze_panes="A2"

for cell in ws2[1]:

    cell.fill=header_fill
    cell.font=header_font
    cell.alignment=Alignment(horizontal="center", vertical="center")


ws2.column_dimensions["A"].width=26
ws2.column_dimensions["B"].width=110

wb.save(out_path)

out_path

'/content/PLD_volatilidades_EGARCH.xlsx'